In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import cv2
import sys
import pandas as pd
from torch.utils.data import Dataset

class FaceDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = int(self.df.iloc[idx]['id'])
        img_path = os.path.join(self.img_dir, f"face-{img_id}.png")
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)["image"]

        return image

In [3]:
sys.path.append("../")
from torch.utils.data import DataLoader
from src.utils.augmentations import get_train_transform
from src.models.diffusion import DiffusionModel


device = "cuda"

dataset = FaceDataset(
    csv_path="../data/train.csv",
    img_dir="../data/processed_64",
    transform=get_train_transform(64)
)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
device = "cuda"
import torch

model = DiffusionModel(device=device, timesteps=1000, img_size=64)
epochs = 200
model.fit(dataloader, epochs=epochs, lr=5e-5)
checkpoint = f"../models/diffusion_model_{epochs}.pth"

torch.save({"model": model.model.state_dict(), "ema_model": model.ema_model.state_dict()}, checkpoint)

Epoch 1/1: 100%|██████████| 71/71 [00:23<00:00,  2.98it/s, loss=0.1039]


In [ ]:
import torchvision

# checkpoint = f"../models/diffusion_model_{epochs}.pth"
# epochs = 1
# model = DiffusionModel(device=device, timesteps=1000, img_size=64)
# ckpt  = torch.load(checkpoint, map_location=device)
# model.model.load_state_dict(ckpt["model"])
# model.ema_model.load_state_dict(ckpt["ema_model"])


# --- Sample ---
samples = model.sample(n=16, use_ema=True)
# --- Rescale [-1, 1] → [0, 1] and save grid ---
samples = (samples.clamp(-1, 1) + 1) / 2
grid    = torchvision.utils.make_grid(samples, nrow=4, padding=2)
torchvision.utils.save_image(grid, f"../outputs/Diffusion_images/generated_{epochs}.png")

Sampling: 100%|██████████| 1000/1000 [00:24<00:00, 40.50it/s]
